In [ ]:
from jax import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import numpyro
from numpyro import distributions as dist
from numpyro import infer
import arviz as az
from pathlib import Path
import plotly.graph_objects as go
import seaborn as sns

from estival.sampling import tools as esamp

from emu_renewal.inputs import DATA_PATH, get_multicountry_df_from_who_data, get_hosp_series_from_owid_data, get_multicountry_df_from_who_data
from emu_renewal.inputs import get_seroprev, filter_seroprev
from emu_renewal.outputs import save_idata, save_spaghetti
from emu_renewal.inputs import get_multivars_country_data, get_row_proportions, VAR_MAP
from emu_renewal.process import CosineMultiCurve
from emu_renewal.distributions import GammaDens
from emu_renewal.renew import MultiStrainModel
from emu_renewal.outputs import get_spaghetti, get_spagh_df_from_dict, get_combined_df, get_df_from_3darray
from emu_renewal.plotting import plot_spaghetti_calib_comparison, plot_proc_comparison, plot_post_prior_comparison
from emu_renewal.calibration import StandardCalib
from emu_renewal.targets import StandardDispTarget

In [ ]:
analysis = "mob"
country = "France"

In [ ]:
# Analysis type
mobility = analysis == "mob"

# WHO indicators
cases_data = get_multicountry_df_from_who_data("New_cases", [country])
deaths_data = get_multicountry_df_from_who_data("New_deaths", [country])

# Hospitalisation data
hosp_data = get_hosp_series_from_owid_data("Daily hospital occupancy", country)

# Variant data
var_data = get_row_proportions(get_multivars_country_data(VAR_MAP, country))
var_data["eu"] = var_data["eu1"] + var_data["eu2"]
var_data = var_data[["eu", "alpha"]]

# Mobility
mob_map = {
    "France": "FR",
}
mob = pd.read_csv(DATA_PATH / f"mobility/{mob_map[country]}_mob_data.csv", index_col=0)
mob.index = pd.to_datetime(mob.index)
non_resi_mob = mob.loc[:, mob.columns!="residential"]
model_mob = non_resi_mob.mean(axis=1).rolling(7).mean().dropna() if mobility else None

In [ ]:
# Specify fixed parameters and get calibration data
proc_update_freq = 14
init_time = 50
pop = 26e6
analysis_start = datetime(2020, 9, 1)
analysis_end = datetime(2021, 6, 15)
data_start = analysis_start + timedelta(14)
init_start = analysis_start - timedelta(init_time)
init_end = analysis_start - timedelta(1)
select_data = cases_data.loc[data_start: analysis_end, country]
select_deaths = deaths_data.loc[data_start: analysis_end, country]
select_vars = var_data.loc[data_start: analysis_end]
hosp_data = hosp_data[data_start: analysis_end: 7]
init_data = cases_data[country].resample("D").asfreq().interpolate().loc[init_start: init_end] / 7.0

In [ ]:
seroprev = filter_seroprev(get_seroprev(), country, analysis_start, analysis_end)
seroprev.index -= timedelta(days=14)

In [ ]:
# Define model and fitter
alpha_seed_times = [datetime(2020, 9, 15), datetime(2021, 10, 20)]
seed_times = [alpha_seed_times]
proc_fitter = CosineMultiCurve()
strains = ["eu", "alpha"]
model_args = [
    pop, 
    analysis_start, 
    analysis_end, 
    proc_update_freq, 
    proc_fitter, 
    GammaDens(), 
    init_time, 
    init_data, 
    GammaDens(), 
    GammaDens(), 
    strains, 
    "eu", 
    seed_times, 
    20.0, 
]
model = MultiStrainModel(*model_args + [model_mob])

In [ ]:
# Define parameter ranges
priors = {
    "gen_mean": dist.TruncatedNormal(7.3, 0.5, low=1.0),
    "gen_sd": dist.TruncatedNormal(3.8, 0.5, low=1.0),
    "cdr": dist.Beta(15, 15), #(16,40)
    "ifr": dist.Beta(3, 200),
    "rt_init": dist.Normal(0.0, 0.25),
    "report_mean": dist.TruncatedNormal(8.0, 0.5, low=1.0),
    "report_sd": dist.TruncatedNormal(3.0, 0.5, low=1.0),
    "death_mean": dist.TruncatedNormal(18.0, 0.5, low=1.0),
    "death_sd": dist.TruncatedNormal(5.0, 0.5, low=1.0),
    "admit_mean": dist.TruncatedNormal(10.0, 1.5, low=1.0),
    "admit_sd": dist.TruncatedNormal(5.0, 0.5, low=1.0),
    "stay_mean": dist.TruncatedNormal(10.0, 1.5, low=1.0),
    "stay_sd": dist.TruncatedNormal(5.0, 0.5, low=1.0),
    "har": dist.Beta(5, 200),
    "shared_dispersion": dist.HalfNormal(0.5),
    "cross_immunity": dist.Uniform(0.01, 0.99),
}

In [ ]:
# Define calibration and calib data
calib_data = {
    "weekly_cases": StandardDispTarget(select_data, weight=40.0),
    "weekly_deaths": StandardDispTarget(select_deaths, weight=50.0),
    "prop_eu": StandardDispTarget(var_data.loc[(datetime(2020, 9, 1) < var_data.index) & (var_data.index < datetime(2021, 7, 1)), "eu"], weight=20.0),
    "seropos": StandardDispTarget(seroprev, weight=20.0),
    "occupancy": StandardDispTarget(hosp_data, weight=20.0),
}
calib = StandardCalib(model, priors, calib_data)

In [ ]:
# Run calibration
analysis_time = datetime.now().strftime("%d%m%Y_%H%M")
kernel = infer.NUTS(calib.calibration, dense_mass=True, init_strategy=calib.custom_init(radius=0.5))
mcmc = infer.MCMC(kernel, num_chains=2, num_samples=100, num_warmup=100)
mcmc.run(random.PRNGKey(1))

In [ ]:
# Grab sample of data from calibrated model outputs
idata = az.from_dict(mcmc.get_samples(True))
idata_sampled = az.extract(idata, num_samples=20)
sample_params = esamp.xarray_to_sampleiterator(idata_sampled)
spaghetti = get_spagh_df_from_dict(get_spaghetti(calib, sample_params))

In [ ]:
plot_spaghetti_calib_comparison(spaghetti, calib.targets, list(calib_data.keys()) + ["process"]).update_layout(title=analysis)

In [ ]:
save_spaghetti(spaghetti, country, analysis, analysis_time)

In [ ]:
idata = az.from_dict(mcmc.get_samples(True))
plot_post_prior_comparison(idata, ["cdr", "stay_mean", "stay_sd", "ifr", "har"], priors);

In [ ]:
save_idata(idata, country, analysis, analysis_time)

In [ ]:
# proc_comp_df = pd.concat([no_mob_spaghetti["process"], mob_spaghetti["process"]], keys=["no_mob", "mob"],axis=1)

In [ ]:
# proc_comparison_fig = go.Figure()
# for i in no_mob_spaghetti["process"]:
#     proc_comparison_fig.add_trace(go.Scatter(x=no_mob_spaghetti["process"].index, y=no_mob_spaghetti["process"][i], line={"color": "orange", "width":0.2}))
# for i in mob_spaghetti["process"]:
#     proc_comparison_fig.add_trace(go.Scatter(x=mob_spaghetti["process"].index, y=mob_spaghetti["process"][i], line={"color": "blue", "width":0.2}))
# proc_comparison_fig.update_layout(showlegend=False, height=500, width=800)

In [ ]:
# mob_updates = pd.DataFrame(mob_sample_params.components["proc"], columns=mob_model.epoch.index_to_dti(mob_model.x_proc_vals)).T
# no_mob_updates = pd.DataFrame(no_mob_sample_params.components["proc"], columns=no_mob_model.epoch.index_to_dti(no_mob_model.x_proc_vals)).T

In [ ]:
# update_comparison_fig = go.Figure()
# for i in no_mob_updates.columns:
#     update_comparison_fig.add_trace(go.Scatter(x=no_mob_updates.index, y=no_mob_updates[i], mode="markers", marker={"color": "orange", "opacity": 0.2, "size": 10.0}))
# for i in mob_updates.columns:
#     update_comparison_fig.add_trace(go.Scatter(x=mob_updates.index, y=mob_updates[i], mode="markers", marker={"color": "blue", "opacity": 0.2, "size": 10.0}))
# update_comparison_fig.update_layout(showlegend=False, height=500, width=800)